# 01 — Data Preparation and Quality Audit

This notebook is the **first research notebook**.

It will:
- load `Main set.xlsx` without modifying the original;
- inspect shape, columns, dates and data types;
- check duplicate dates, missing values and calendar gaps;
- validate TT buy/sell/mid consistency for USD, GBP, EUR and JPY;
- flag `buy > sell`, negative spreads and mid-rate inconsistencies;
- inspect weekday/weekend repetition and unchanged quotes;
- summarize spreads, returns, large movements and rolling volatility;
- inspect cross-currency return relationships;
- inspect how frequently each macro/global variable changes;
- save audit tables and a quality-flagged dataset.

**No modelling is performed here.**

Real crisis-period observations are not removed simply because they look extreme.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

CURRENCIES = ["USD", "GBP", "EUR", "JPY"]

print("Environment ready.")

In [ ]:
# Find the Research project root safely.

def find_project_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]

    for root in candidates:
        if (root / "data" / "raw" / "Main set.xlsx").exists():
            return root.resolve()

    raise FileNotFoundError(
        "Could not find data/raw/Main set.xlsx. "
        "Keep the file inside Research/data/raw/"
    )

PROJECT_ROOT = find_project_root()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

for p in [PROCESSED_DIR, TABLE_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATA_PATH = RAW_DIR / "Main set.xlsx"

print("Project root :", PROJECT_ROOT)
print("Dataset path :", DATA_PATH)
print("Tables       :", TABLE_DIR)
print("Figures      :", FIGURE_DIR)

In [ ]:
# Load the original Excel workbook.

xls = pd.ExcelFile(DATA_PATH)

print("Workbook sheets:", xls.sheet_names)

df_raw = pd.read_excel(DATA_PATH, sheet_name=xls.sheet_names[0])

print("Raw shape:", df_raw.shape)
display(df_raw.head())

## Structural overview

Expected core columns are:
- `Date`
- USD/GBP/EUR/JPY buy, sell and mid rates
- macroeconomic and global-market variables.

In [ ]:
print("Rows   :", len(df_raw))
print("Columns:", len(df_raw.columns))

column_overview = pd.DataFrame({
    "Column": df_raw.columns,
    "Data_Type": [str(x) for x in df_raw.dtypes],
    "Non_Null": [df_raw[c].notna().sum() for c in df_raw.columns],
    "Unique_Values": [df_raw[c].nunique(dropna=True) for c in df_raw.columns],
})

display(column_overview)
column_overview.to_csv(TABLE_DIR / "01_column_overview.csv", index=False)

In [ ]:
# Required-column check.

required = ["Date"]
for cur in CURRENCIES:
    required += [f"{cur}_buy", f"{cur}_sell", f"{cur}_mid"]

missing_required = [c for c in required if c not in df_raw.columns]

if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

print("PASS — all required TT columns are present.")

## Date integrity

We check:
- valid date conversion;
- chronological order;
- earliest/latest dates;
- duplicate dates;
- missing calendar dates.

In [ ]:
df = df_raw.copy()
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

invalid_dates = int(df["Date"].isna().sum())
print("Invalid/missing dates:", invalid_dates)

if invalid_dates:
    display(df[df["Date"].isna()].head(20))

df = df.sort_values("Date").reset_index(drop=True)

print("Start date:", df["Date"].min())
print("End date  :", df["Date"].max())
print("Sorted    :", df["Date"].is_monotonic_increasing)

In [ ]:
duplicate_date_mask = df["Date"].duplicated(keep=False)
duplicate_row_mask = df.duplicated(keep=False)

n_duplicate_dates = int(duplicate_date_mask.sum())
n_duplicate_rows = int(duplicate_row_mask.sum())

print("Rows belonging to duplicate dates:", n_duplicate_dates)
print("Fully duplicated rows             :", n_duplicate_rows)

if n_duplicate_dates:
    display(df.loc[duplicate_date_mask].sort_values("Date").head(50))
else:
    print("PASS — no duplicate dates.")

In [ ]:
full_calendar = pd.date_range(df["Date"].min(), df["Date"].max(), freq="D")
observed_dates = pd.DatetimeIndex(df["Date"].dropna().unique())
missing_calendar_dates = full_calendar.difference(observed_dates)

print("Expected calendar days:", len(full_calendar))
print("Observed dates        :", len(observed_dates))
print("Missing calendar dates:", len(missing_calendar_dates))

if len(missing_calendar_dates):
    display(pd.DataFrame({"Missing_Date": missing_calendar_dates}).head(30))
else:
    print("PASS — complete daily calendar.")

## Missing-value audit

We only **report** missingness here. No automatic imputation is performed.

In [ ]:
missing_summary = pd.DataFrame({
    "Column": df.columns,
    "Missing_Count": [df[c].isna().sum() for c in df.columns],
    "Missing_Percent": [100 * df[c].isna().mean() for c in df.columns],
}).sort_values(
    ["Missing_Count", "Column"],
    ascending=[False, True]
).reset_index(drop=True)

display(missing_summary)

missing_summary.to_csv(
    TABLE_DIR / "01_missing_value_summary.csv",
    index=False
)

print("Total missing cells:", int(df.isna().sum().sum()))

## Descriptive statistics

These summaries are descriptive only. Extreme values are not automatically treated as errors.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

numeric_summary = df[numeric_cols].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

numeric_summary["missing"] = df[numeric_cols].isna().sum()
numeric_summary["unique"] = df[numeric_cols].nunique(dropna=True)

display(numeric_summary)

numeric_summary.to_csv(
    TABLE_DIR / "01_numeric_descriptive_summary.csv"
)

# TT-rate integrity

For each currency:

\[
Mid_t \approx \frac{Buy_t + Sell_t}{2}
\]

and normally:

\[
Sell_t \ge Buy_t
\]

Violations are flagged for investigation, not silently corrected.

In [ ]:
tt_integrity_rows = []

for cur in CURRENCIES:
    buy = pd.to_numeric(df[f"{cur}_buy"], errors="coerce")
    sell = pd.to_numeric(df[f"{cur}_sell"], errors="coerce")
    mid = pd.to_numeric(df[f"{cur}_mid"], errors="coerce")

    computed_mid = (buy + sell) / 2
    mid_abs_error = (mid - computed_mid).abs()
    spread = sell - buy

    tt_integrity_rows.append({
        "Currency": cur,
        "Valid_Rows": int((buy.notna() & sell.notna() & mid.notna()).sum()),
        "Buy_Greater_Than_Sell": int((buy > sell).sum()),
        "Negative_Spread_Count": int((spread < 0).sum()),
        "Zero_Spread_Count": int((spread == 0).sum()),
        "Max_Mid_Arithmetic_Error": float(mid_abs_error.max()),
        "Mean_Mid_Arithmetic_Error": float(mid_abs_error.mean()),
        "Mid_Error_Over_1e-6": int((mid_abs_error > 1e-6).sum()),
    })

tt_integrity = pd.DataFrame(tt_integrity_rows)

display(tt_integrity)

tt_integrity.to_csv(
    TABLE_DIR / "01_tt_integrity_summary.csv",
    index=False
)

In [ ]:
# Show every buy > sell anomaly.

anomaly_frames = []

for cur in CURRENCIES:
    mask = df[f"{cur}_buy"] > df[f"{cur}_sell"]

    if mask.any():
        temp = df.loc[
            mask,
            ["Date", f"{cur}_buy", f"{cur}_sell", f"{cur}_mid"]
        ].copy()

        temp.insert(1, "Currency", cur)
        temp["Spread"] = temp[f"{cur}_sell"] - temp[f"{cur}_buy"]
        anomaly_frames.append(temp)

if anomaly_frames:
    buy_sell_anomalies = pd.concat(anomaly_frames, ignore_index=True)
    display(buy_sell_anomalies)

    buy_sell_anomalies.to_csv(
        TABLE_DIR / "01_buy_greater_than_sell_anomalies.csv",
        index=False
    )
else:
    buy_sell_anomalies = pd.DataFrame()
    print("PASS — no buy > sell anomalies.")

In [ ]:
# Add quality flags while preserving original values.

df_audit = df.copy()

for cur in CURRENCIES:
    buy = df_audit[f"{cur}_buy"]
    sell = df_audit[f"{cur}_sell"]
    mid = df_audit[f"{cur}_mid"]

    df_audit[f"{cur}_spread"] = sell - buy
    df_audit[f"{cur}_spread_pct"] = (
        (sell - buy) / mid.replace(0, np.nan)
    ) * 100

    df_audit[f"{cur}_computed_mid"] = (buy + sell) / 2
    df_audit[f"{cur}_mid_abs_error"] = (
        mid - df_audit[f"{cur}_computed_mid"]
    ).abs()

    df_audit[f"{cur}_flag_buy_gt_sell"] = buy > sell
    df_audit[f"{cur}_flag_mid_mismatch"] = (
        df_audit[f"{cur}_mid_abs_error"] > 1e-6
    )

print("Quality flags created. Original TT values were not overwritten.")

## Weekday/weekend behaviour

Daily TT datasets often repeat bank quotations on weekends.  
We inspect this before deciding the modelling calendar.

In [ ]:
df_audit["Day_Name"] = df_audit["Date"].dt.day_name()
df_audit["Day_of_Week"] = df_audit["Date"].dt.dayofweek
df_audit["Is_Weekend"] = df_audit["Day_of_Week"].isin([5, 6])

calendar_summary = (
    df_audit.groupby("Is_Weekend")
    .size()
    .rename("Rows")
    .reset_index()
)

calendar_summary["Calendar_Type"] = np.where(
    calendar_summary["Is_Weekend"],
    "Weekend",
    "Weekday"
)

display(calendar_summary[["Calendar_Type", "Rows"]])

In [ ]:
unchanged_rows = []

for cur in CURRENCIES:
    col = f"{cur}_mid"
    unchanged = df_audit[col].eq(df_audit[col].shift(1))

    groups = [
        ("All", pd.Series(True, index=df_audit.index)),
        ("Weekday", ~df_audit["Is_Weekend"]),
        ("Weekend", df_audit["Is_Weekend"]),
    ]

    for label, mask in groups:
        valid = mask & df_audit[col].notna()

        unchanged_rows.append({
            "Currency": cur,
            "Calendar_Type": label,
            "Observations": int(valid.sum()),
            "Unchanged_Count": int((unchanged & valid).sum()),
            "Unchanged_Percent": (
                100 * (unchanged & valid).sum() / valid.sum()
                if valid.sum() else np.nan
            )
        })

unchanged_summary = pd.DataFrame(unchanged_rows)

display(unchanged_summary)

unchanged_summary.to_csv(
    TABLE_DIR / "01_unchanged_quote_summary.csv",
    index=False
)

## TT spread analysis

Spread is important later for micro-FX decision support.

In [ ]:
spread_rows = []

for cur in CURRENCIES:
    s = df_audit[f"{cur}_spread"]
    sp = df_audit[f"{cur}_spread_pct"]

    spread_rows.append({
        "Currency": cur,
        "Mean_Spread": s.mean(),
        "Median_Spread": s.median(),
        "Min_Spread": s.min(),
        "Max_Spread": s.max(),
        "Mean_Spread_Percent": sp.mean(),
        "Median_Spread_Percent": sp.median(),
    })

spread_summary = pd.DataFrame(spread_rows)

display(spread_summary)

spread_summary.to_csv(
    TABLE_DIR / "01_tt_spread_summary.csv",
    index=False
)

## TT mid-rate history

In [ ]:
for cur in CURRENCIES:
    plt.figure(figsize=(12, 4))
    plt.plot(df_audit["Date"], df_audit[f"{cur}_mid"], linewidth=1)
    plt.title(f"{cur}/LKR TT Mid Rate — Full History")
    plt.xlabel("Date")
    plt.ylabel(f"{cur}/LKR TT Mid")
    plt.tight_layout()

    plt.savefig(
        FIGURE_DIR / f"01_{cur}_tt_mid_history.png",
        dpi=150,
        bbox_inches="tight"
    )
    plt.show()

## Daily log-return insights

\[
r_t = \ln(P_t/P_{t-1})
\]

Large movements are flagged for review only. They are not automatically deleted.

In [ ]:
for cur in CURRENCIES:
    mid = df_audit[f"{cur}_mid"]
    df_audit[f"{cur}_log_return"] = np.log(mid / mid.shift(1))

return_rows = []

for cur in CURRENCIES:
    r = df_audit[f"{cur}_log_return"]

    return_rows.append({
        "Currency": cur,
        "Mean": r.mean(),
        "Std": r.std(),
        "Min": r.min(),
        "Max": r.max(),
        "Median": r.median(),
        "Abs_95pct": r.abs().quantile(0.95),
        "Abs_99pct": r.abs().quantile(0.99),
    })

return_summary = pd.DataFrame(return_rows)

display(return_summary)

return_summary.to_csv(
    TABLE_DIR / "01_log_return_summary.csv",
    index=False
)

In [ ]:
# Robust large-movement flags using median absolute deviation.

large_move_frames = []

for cur in CURRENCIES:
    r = df_audit[f"{cur}_log_return"]

    median = r.median()
    mad = np.nanmedian(np.abs(r - median))

    if mad == 0 or not np.isfinite(mad):
        continue

    robust_z = 0.6745 * (r - median) / mad
    df_audit[f"{cur}_return_robust_z"] = robust_z

    mask = robust_z.abs() > 5

    if mask.any():
        temp = df_audit.loc[
            mask,
            ["Date", f"{cur}_mid", f"{cur}_log_return", f"{cur}_return_robust_z"]
        ].copy()

        temp.insert(1, "Currency", cur)
        large_move_frames.append(temp)

if large_move_frames:
    large_movements = pd.concat(
        large_move_frames,
        ignore_index=True
    ).sort_values(["Date", "Currency"])

    print("Large movements flagged:", len(large_movements))
    display(large_movements.head(50))

    large_movements.to_csv(
        TABLE_DIR / "01_large_return_movements_for_review.csv",
        index=False
    )
else:
    large_movements = pd.DataFrame()
    print("No large movements exceeded the review threshold.")

## Rolling volatility

In [ ]:
for cur in CURRENCIES:
    r = df_audit[f"{cur}_log_return"]
    rolling_vol = r.rolling(30, min_periods=15).std() * np.sqrt(252)

    plt.figure(figsize=(12, 4))
    plt.plot(df_audit["Date"], rolling_vol, linewidth=1)
    plt.title(f"{cur}/LKR — 30-Observation Rolling Annualized Volatility")
    plt.xlabel("Date")
    plt.ylabel("Annualized volatility")
    plt.tight_layout()

    plt.savefig(
        FIGURE_DIR / f"01_{cur}_rolling_volatility_30.png",
        dpi=150,
        bbox_inches="tight"
    )
    plt.show()

## Cross-currency return relationships

This is an initial descriptive view only. Formal rolling correlation/regime analysis comes later.

In [ ]:
return_cols = [f"{cur}_log_return" for cur in CURRENCIES]
return_corr = df_audit[return_cols].corr()

display(return_corr)

return_corr.to_csv(
    TABLE_DIR / "01_currency_return_correlations.csv"
)

plt.figure(figsize=(6, 5))
plt.imshow(return_corr.values, aspect="auto")
plt.xticks(range(len(CURRENCIES)), CURRENCIES)
plt.yticks(range(len(CURRENCIES)), CURRENCIES)
plt.colorbar(label="Correlation")
plt.title("Correlation of Daily TT Log Returns")

for i in range(len(CURRENCIES)):
    for j in range(len(CURRENCIES)):
        plt.text(
            j, i,
            f"{return_corr.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "01_currency_return_correlation_matrix.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()

## Variable update-frequency audit

This helps identify daily versus slowly changing variables.  
It does **not** yet establish exact publication timing; that will be handled in the point-in-time feature notebook.

In [ ]:
update_rows = []

for c in numeric_cols:
    s = pd.to_numeric(df_audit[c], errors="coerce")

    changed = (
        s.ne(s.shift(1))
        & s.notna()
        & s.shift(1).notna()
    )

    denom = max(1, int(s.notna().sum()) - 1)

    update_rows.append({
        "Column": c,
        "Non_Missing": int(s.notna().sum()),
        "Unique_Values": int(s.nunique(dropna=True)),
        "Observed_Changes": int(changed.sum()),
        "Change_Percent_of_Rows": 100 * changed.sum() / denom
    })

update_frequency = pd.DataFrame(update_rows).sort_values(
    "Change_Percent_of_Rows",
    ascending=False
)

display(update_frequency)

update_frequency.to_csv(
    TABLE_DIR / "01_variable_update_frequency.csv",
    index=False
)

## Yearly TT summary and crisis-window overview

In [ ]:
df_audit["Year_Audit"] = df_audit["Date"].dt.year

yearly_rows = []

for cur in CURRENCIES:
    temp = (
        df_audit
        .groupby("Year_Audit")[f"{cur}_mid"]
        .agg(["count", "mean", "median", "min", "max", "std"])
        .reset_index()
    )
    temp.insert(1, "Currency", cur)
    yearly_rows.append(temp)

yearly_tt_summary = pd.concat(yearly_rows, ignore_index=True)

display(yearly_tt_summary)

yearly_tt_summary.to_csv(
    TABLE_DIR / "01_yearly_tt_summary.csv",
    index=False
)

In [ ]:
crisis_window = df_audit[
    df_audit["Date"].between("2021-01-01", "2023-12-31")
].copy()

crisis_rows = []

for cur in CURRENCIES:
    for year, g in crisis_window.groupby(crisis_window["Date"].dt.year):
        crisis_rows.append({
            "Currency": cur,
            "Year": int(year),
            "Start_Mid": g[f"{cur}_mid"].iloc[0],
            "End_Mid": g[f"{cur}_mid"].iloc[-1],
            "Min_Mid": g[f"{cur}_mid"].min(),
            "Max_Mid": g[f"{cur}_mid"].max(),
            "Log_Return_Std": g[f"{cur}_log_return"].std(),
        })

crisis_summary = pd.DataFrame(crisis_rows)

display(crisis_summary)

crisis_summary.to_csv(
    TABLE_DIR / "01_crisis_window_2021_2023_summary.csv",
    index=False
)

## Data-quality scorecard

In [ ]:
quality_scorecard = pd.DataFrame([
    {
        "Check": "Dataset loaded",
        "Status": "PASS",
        "Evidence": f"{len(df):,} rows × {len(df.columns)} raw columns"
    },
    {
        "Check": "Valid dates",
        "Status": "PASS" if invalid_dates == 0 else "REVIEW",
        "Evidence": f"{invalid_dates} invalid/missing dates"
    },
    {
        "Check": "Duplicate dates",
        "Status": "PASS" if n_duplicate_dates == 0 else "REVIEW",
        "Evidence": f"{n_duplicate_dates} rows belong to duplicated dates"
    },
    {
        "Check": "Complete daily calendar",
        "Status": "PASS" if len(missing_calendar_dates) == 0 else "REVIEW",
        "Evidence": f"{len(missing_calendar_dates)} missing dates"
    },
    {
        "Check": "Missing values",
        "Status": "PASS" if df.isna().sum().sum() == 0 else "REVIEW",
        "Evidence": f"{int(df.isna().sum().sum())} missing cells"
    },
    {
        "Check": "TT mid arithmetic",
        "Status": "PASS" if tt_integrity["Mid_Error_Over_1e-6"].sum() == 0 else "REVIEW",
        "Evidence": f"{int(tt_integrity['Mid_Error_Over_1e-6'].sum())} mid-rate mismatches"
    },
    {
        "Check": "Buy <= sell",
        "Status": "PASS" if tt_integrity["Buy_Greater_Than_Sell"].sum() == 0 else "REVIEW",
        "Evidence": f"{int(tt_integrity['Buy_Greater_Than_Sell'].sum())} buy > sell records"
    },
])

display(quality_scorecard)

quality_scorecard.to_csv(
    TABLE_DIR / "01_data_quality_scorecard.csv",
    index=False
)

## Save the quality-flagged dataset

This file is an **audit dataset**, not yet the final modelling dataset.

In [ ]:
audit_path = PROCESSED_DIR / "01_quality_flagged_daily_dataset.csv"

df_audit.to_csv(audit_path, index=False)

print("Saved:", audit_path)

In [ ]:
print("=" * 72)
print("NOTEBOOK 01 — DATA PREPARATION & QUALITY AUDIT COMPLETE")
print("=" * 72)

print(f"Observations : {len(df):,}")
print(f"Raw variables: {len(df.columns):,}")
print("Date range   :", df["Date"].min().date(), "to", df["Date"].max().date())
print("Duplicate-date rows:", n_duplicate_dates)
print("Missing calendar dates:", len(missing_calendar_dates))
print("Missing cells:", int(df.isna().sum().sum()))
print("Buy > sell anomalies:", int(tt_integrity["Buy_Greater_Than_Sell"].sum()))
print("Mid arithmetic mismatches:", int(tt_integrity["Mid_Error_Over_1e-6"].sum()))

print("\nIMPORTANT:")
print("Large exchange-rate movements were flagged only — not removed.")
print("The original Main set.xlsx was not overwritten.")
print("\nNext notebook: 02_Targets_and_Point_in_Time_Features.ipynb")